#  Predicting Stroke on New Patient Data

## Import all necessary libraries

In [1]:
import joblib
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Define custom feature engineering functions
def categorize_features(df_in):
    df = df_in.copy()
    
    # 1. High Glucose Flag
    df['is_high_glucose'] = (df['avg_glucose_level'] > 150).astype(int)
    
    # 2. BMI Categories
    bins = [0, 18.5, 25, 30, 100]
    labels = ['Underweight', 'Normal', 'Overweight', 'Obese']
    # Use fillna to handle missing BMI values temporarily before the imputer catches them
    df['bmi_category'] = pd.cut(df['bmi'].fillna(df['bmi'].median()), bins=bins, labels=labels)
    
    # 3. Age Categories
    age_bins = [0, 18, 65, 150]
    age_labels = ['Child', 'Adult', 'Senior']
    df['age_category'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)
    
    # 4. Health Risk Score (Sum of risk factors)
    df['health_risk_score'] = (
        (df['age_category'] == 'Senior').astype(int) +
        df['hypertension'] +
        df['heart_disease'] +
        (df['bmi_category'].isin(['Overweight', 'Obese'])).astype(int) +
        (df['smoking_status'] == 'smokes').astype(int)
    )
    
    return df

### Load model

In [7]:
try:
    production_model = joblib.load(r'C:\Users\ADMIN\Desktop\Logic Mojo_AIML Jan26\AI\Assignment_05\stroke_pipeline_v1.pkl')
    print("Model successfully loaded and ready for predictions!")
except FileNotFoundError:
    print(r"Error: Could not find 'C:\Users\ADMIN\Desktop\Logic Mojo_AIML Jan26\AI\Assignment_05\stroke_pipeline_v1.pkl'.")

Model successfully loaded and ready for predictions!


### Simulating New Patient Data

In [8]:
new_patients = pd.DataFrame([
    {
        'gender': 'Male',
        'age': 88.0,
        'hypertension': 1,
        'heart_disease': 1,
        'ever_married': 'Yes',
        'work_type': 'Private',
        'Residence_type': 'Urban',
        'avg_glucose_level': 210.5,
        'bmi': 36.6,
        'smoking_status': 'smokes'
    },
    {
        # Patient B: Healthy Profile (Young, active, normal vitals)
        'gender': 'Female',
        'age': 19.0,
        'hypertension': 0,
        'heart_disease': 0,
        'ever_married': 'No',
        'work_type': 'Private',
        'Residence_type': 'Rural',
        'avg_glucose_level': 85.0,
        'bmi': 22.4, # Healthy BMI
        'smoking_status': 'never smoked'
    },
    {
        # Patient C: Missing Data Profile (To prove our Imputer works!)
        'gender': 'Male',
        'age': 55.0,
        'hypertension': 0,
        'heart_disease': 0,
        'ever_married': 'Yes',
        'work_type': 'Self-employed',
        'Residence_type': 'Suburban', # Note: Our pipeline might not have seen 'Suburban' exactly, but it should handle it gracefully or ignore it.
        'avg_glucose_level': 130.0,
        'bmi': None, # Missing BMI! Our pipeline's SimpleImputer will fill this with the median it learned during training.
        'smoking_status': 'Unknown'
    }
])

display(new_patients)

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status
0,Male,88.0,1,1,Yes,Private,Urban,210.5,36.6,smokes
1,Female,19.0,0,0,No,Private,Rural,85.0,22.4,never smoked
2,Male,55.0,0,0,Yes,Self-employed,Suburban,130.0,NaN,Unknown


### One-Click Inference

In [11]:
predictions = production_model.predict(new_patients)
predictions

array([1, 0, 0])

In [12]:
probabilities = production_model.predict_proba(new_patients)[:, 1]
probabilities

array([0.7066619 , 0.18683541, 0.4915467 ], dtype=float32)

In [13]:
results_df = new_patients.copy()
results_df['Predicted_Stroke_Risk'] = ['High Risk' if p == 1 else 'Low Risk' for p in predictions]
results_df['Probability_Score'] = [f"{prob * 100:.1f}%" for prob in probabilities]

# Show the doctor the final output!
display(results_df[['age', 'avg_glucose_level', 'Predicted_Stroke_Risk', 'Probability_Score']])

,age,avg_glucose_level,Predicted_Stroke_Risk,Probability_Score
0,88.0,210.5,High Risk,70.7%
1,19.0,85.0,Low Risk,18.7%
2,55.0,130.0,Low Risk,49.2%
